# A Guide to how this program functions:


1.   We get the location the user wants to search properties from (State/City)

2.   We get the type of property theyre interested in
3.   We print out a brief panda of every search result initially found
4.   To help the user, we make a bar chart of the number of properties per price range that fit their search so far. So they can get an idea of what the market looks like
5.   Once they have this info, we get their price range inputs
6.   The panda df is narrowed based on the price range and displayed
7.   Now that we have a smaller df, we go through and web scrape the number of bedrooms, bathrooms, and sq. ft. for each property
8.   Then we get the max number of bedrooms and bathrooms the user would want. After, we slice the df some more based on those inputs and display it
9.   Now we need the description for each property. I saved this for now because the process takes a bit, so I only wanted to fetch the descriptions when we needed to. This process uses selenium's webdriver because some descriptions are hidden under a 'See more' dropdown click
10.  Now that we have each property's description, we allow the user to enter as many keywords as they want. These keywords are the types of features that they want in a property
11.  We then compare the user's keywords to each property's description, calculate the cosine similarity, and sort the df by highest cosine similarity scores
12.  Time to take the top 3 matches found and display the property info for each one. We nicely display all of the info for each row in the top 3 + we webscrape a few pictures of each property, and display those too
13.  We've made it to the end! Now we have a real estate program that can help the user find a property that fits them in any city in the United States

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

def get_user_search_info():
  """
  Gets the state, city, and property type that the user wants to search for.

  Returns
  -------
  The information that the user entered.

  """
  states_abr_list = ["AK", "AL", "AR", "AZ", "CA", "CO", "CT", "DE", "FL", "GA", "HI", "IA",
    "ID", "IL", "IN", "KS", "KY", "LA", "MA", "MD", "ME", "MI", "MN", "MO",
    "MS", "MT", "NC", "ND", "NE", "NH", "NJ", "NM", "NV", "NY", "OH", "OK",
    "OR", "PA", "RI", "SC", "SD", "TN", "TX", "UT", "VA", "VT", "WA", "WI",
    "WV", "WY"]

  while True:
    users_state = input('Enter the 2 letter abbreviation of the state you want to see houses for: ')
    users_state = users_state.upper()
    print(users_state)
    if users_state not in states_abr_list:
      print('Please enter a valid state abbreviation')
      continue

    users_city = input(f'Enter the city in {users_state} you want to see houses for: ')
    users_city = users_city.replace(' ', '-')

    property_type = input('''Enter the number that corresponds to the type of home you\'re looking for:
    1) Single Family Home
    2) Condo
    3) Townhouse
    4) Duplex
    5) Mobile Home
    ''')
    # property_pref is set to what homesfinder.com adds to the URL
    # For a Single Family Home they add SFH to the URL, etc
    property_pref = ''
    if property_type == '1':
      property_pref = 'SFH'
    elif property_type == '2':
      property_pref = 'CONDOUNIT'
    elif property_type == '3':
      property_pref = 'TOWNHOUSE'
    elif property_type == '4':
      property_pref = 'DUPLEX'
    elif property_type == '5':
      property_pref = 'MOBILEHOME'
    else:
      print('Please enter a valid number')
      continue
    url_tags = f'{users_state}/{users_city}?propertyTypes={property_pref}'
    url = 'https://homefinder.com/homes-for-sale/' + url_tags

    response = requests.get(url)
    # Make sure we get a proper webpage. Otherwise the location didnt exist and we need to re-loop
    if response.status_code == 200:
      break
    else:
      print('Please enter a valid location')
      continue

  return (url, users_state, users_city, property_pref)


In [38]:
import re

# Call our method to get to a working webpage with all the initial search results
# Then we want a quick overview of each property: price, type, direct link
prices_list = []
house_type = []
href_links = []

users_info = get_user_search_info()
site_url = users_info[0]
print('site_url = ' + site_url)

html = requests.get(site_url).text

soup = BeautifulSoup(html, "html.parser")

element = soup.select_one('[data-testid="total-results"]')

if element:
    text = element.get_text(strip=True)
    print(text)

# Need to figure out how many pages our result has
# There are 22 searches per page, so we can calculate the num of pages from the total results
num_of_for_sale = element.get_text(strip=True)
num_of_for_sale = num_of_for_sale.replace(',', '')
print(num_of_for_sale)
#num_of_for_sale = re.findall('(\d+)', num_of_for_sale.text)
# Had to do a join in case we had a search that goes over 1,000: the comma turned it into a multiple numbered list
#num_of_for_sale = ''.join(num_of_for_sale)
#Snum_of_for_sale.replace('', ',')
RESULTS_PER_PAGE = 42
num_page_results = int(num_of_for_sale) / 42
num_page_results = int(num_page_results + 1)
print('number of initial search pages to go through: ' + str(num_page_results))
print('Loading the search results...')

# Need to update the URL to add the page number to it. First page has no page number
for i in range(num_page_results):
  if i == 0:
    url = site_url
  else:
    url = f'{site_url}&page={i + 1}'

  doc = requests.get(url)
  page = BeautifulSoup(doc.text, 'html.parser')
  house_links = page.find_all('a', class_='listing-tile d-flex flex-column')
  for h in house_links:
    full_link = 'https://homefinder.com' + h.get('href')
    href_links.append(full_link)

  prices = page.find_all('div', class_='h4 text-primary mb-0')
  for p in prices:
    # This tag gives us both the price and type, once we split it
    full_string = p.text
    full_string = full_string.replace('\n', '')
    full_string = full_string.strip()
    split_string = full_string.split()
    prices_list.append(split_string[0])
    house_type.append(split_string[1])

data = {'Price': prices_list, 'Type': house_type, 'Link': href_links}
df = pd.DataFrame(data)

# Drop all the rows that dont match the property type
# There were SFH's under the Condo results
property_type = users_info[3]
if property_type == 'SFH':
  df = df[df['Type'] == 'House']
elif property_type == 'CONDOUNIT':
  df = df[df['Type'] == 'Condo']
elif property_type == 'TOWNHOUSE':
  df = df[df['Type'] == 'Townhouse']
elif property_type == 'DUPLEX':
  df = df[df['Type'] == 'Duplex']

print('Number of properties found that match your search: ' + str(len(df)))
# Left Mobile Homes out of this because their type is listed as house
print(df.to_string())


Enter the 2 letter abbreviation of the state you want to see houses for: CA
CA
Enter the city in CA you want to see houses for: San Diego
Enter the number that corresponds to the type of home you're looking for:
    1) Single Family Home
    2) Condo
    3) Townhouse
    4) Duplex
    5) Mobile Home
    1
site_url = https://homefinder.com/homes-for-sale/CA/San-Diego?propertyTypes=SFH
3,788
3788
number of initial search pages to go through: 91
Loading the search results...


KeyboardInterrupt: 

In [36]:
url = "https://homefinder.com/realestateandhomes-search/San-Diego_CA"
html = requests.get(url).text

soup = BeautifulSoup(html, "html.parser")

element = soup.select_one('[data-testid="total-results"]')

if element:
    text = element.get_text(strip=True)
    text = text.replace(',', '')
    print(text)


3788


In [ ]:
# Create a prices_by_int column for calculations later
prices_list = []
for i, row in df.iterrows():
  price = row['Price']
  price = price.replace('$', '')
  price = price.replace(',', '')
  prices_list.append(int(price))

df['Prices_by_int'] = prices_list

# Go through the column and count how many properties are in each price range
count_of_prices = {'Under200k': 0, '200k-400k': 0, '400k-600k': 0, '600k-800k': 0, '800k-1m': 0, '1m-2m': 0, '2m-3m': 0, 'Over3m':0}
for p in prices_list:
  if p <= 200000:
    count_of_prices['Under200k'] += 1
  elif p > 200000 and p <= 400000:
    count_of_prices['200k-400k'] += 1
  elif p > 400000 and p <= 600000:
    count_of_prices['400k-600k'] += 1
  elif p > 600000 and p <= 800000:
    count_of_prices['600k-800k'] += 1
  elif p > 800000 and p <= 1000000:
    count_of_prices['800k-1m'] += 1
  elif p > 1000000 and p <= 2000000:
    count_of_prices['1m-2m'] += 1
  elif p > 2000000 and p <= 3000000:
    count_of_prices['2m-3m'] += 1
  else:
    count_of_prices['Over3m'] += 1


In [ ]:
# Create a bar chart of the price ranges/counts that the search found
range_labels = []
range_counts = []
for key, value in count_of_prices.items():
  range_labels.append(key)
  range_counts.append(value)

import matplotlib.pyplot as plt
plt.figure(figsize=(13,5))
plt.title('Count of Properties per Price Range', fontsize=15, fontstyle='italic')
plt.xlabel('Price Range')
plt.ylabel('Number of Properties')
plt.grid(True, axis='y', color='0.5', linestyle='--')
plt.bar(range_labels, range_counts, edgecolor='black', linewidth=1)
plt.show()

In [ ]:
def get_price_range():
  """
  Gets the price range from the user

  Returns
  -------
  The minimum price and the maximum price that the user entered.

  """
  while True:
    min_price = input('Enter the minimum price for your price range: ')
    min_price = min_price.replace('$', '')
    min_price = min_price.replace(',', '')
    max_price = input('Enter the maximum price for your price range: ')
    max_price = max_price.replace('$', '')
    max_price = max_price.replace(',', '')
    if min_price.isdigit() and max_price.isdigit():
      if int(min_price) < int(max_price):
        break
      else:
        print('Please enter a valid price range')
        continue

  return int(min_price), int(max_price)

price_points = get_price_range()

# Slice the df to get rid of properties that are outside the user's price range
df = df.loc[(df['Prices_by_int'] >= price_points[0]) & (df['Prices_by_int'] <= price_points[1])]


df = df.drop(columns=['Prices_by_int'])
df = df.sort_values(by=['Price'])

def print_panda(df):
  """
  Nice prints a panda DataFrame using tabulate.

  Parameters
  ----------
  df : The panda to print.

  """
  from tabulate import tabulate
  df_cols = df.columns.tolist()
  df = df.reset_index(drop=True)
  print(tabulate(df, headers=df_cols, tablefmt='psql'))

print_panda(df)

In [ ]:
# Find num of bedrooms, num of bathrooms, and sq ft total
bedrooms_list = []
bathrooms_list = []
sq_ft_list = []

for i, row in df.iterrows():
  url = row['Link']
  doc = requests.get(url)
  page = BeautifulSoup(doc.text, 'html.parser')
  details = page.find('ul', class_='bullet-list-horizontal list-unstyled')

  # Had some random results where a property detail linked to many other properties instead of linking to one property.
  # This led to details for it equaling None
  # Assign it N/A values and our panda will skip it
  if details == None:
    bedrooms_list.append('N/A')
    bathrooms_list.append('N/A')
    sq_ft_list.append('N/A')
    continue
  else:
    details = details.text.strip()
    if details == '':
      bedrooms_list.append('N/A')
      bathrooms_list.append('N/A')
      sq_ft_list.append('N/A')
    else:
      split_details = details.split()
      print(split_details)
      # Standard format of our split would have num, Beds, num, Baths, num, sq ft
      # However, not all properties have all 3 datapoints. Need to verify
      if 'Beds' in split_details:
        beds_index = split_details.index('Beds')
        bedrooms_list.append(split_details[beds_index - 1])
      elif 'Bd' in split_details:
        beds_index = split_details.index('Bd')
        bedrooms_list.append(split_details[beds_index - 1])
      else:
        bedrooms_list.append('N/A')

      if 'Baths' in split_details:
        baths_index = split_details.index('Baths')
        bathrooms_list.append(split_details[baths_index - 1])
      elif 'Bath' in split_details:
        baths_index = split_details.index('Bath')
        bathrooms_list.append(split_details[baths_index - 1])
      else:
        bathrooms_list.append('N/A')

      if 'sqft' in split_details:
        sq_ft_index = split_details.index('sqft')
        sq_ft_list.append(split_details[sq_ft_index - 1])
      elif 'SqFt' in split_details:
        sq_ft_index = split_details.index('SqFt')
        sq_ft_list.append(split_details[sq_ft_index - 1])
      elif 'Sqft' in split_details:
        sq_ft_index = split_details.index('Sqft')
        sq_ft_list.append(split_details[sq_ft_index - 1])
      else:
        sq_ft_list.append('N/A')

# Add the new data as columns to the df
df.loc[:, '# Bedrooms'] = bedrooms_list
df.loc[:, '# Bathrooms'] = bathrooms_list
df.loc[:, 'Sq. Ft.'] = sq_ft_list

# Drop any N/A bedrooms or bathrooms. N/A sq ft is okay
for i, row in df.iterrows():
  bedrooms = row['# Bedrooms']
  bathrooms = row['# Bathrooms']
  if bedrooms == 'N/A' or bathrooms == 'N/A':
    df.drop(i, inplace=True)

print_panda(df)

In [ ]:
def get_room_ranges():
  """
  Gets the inputs for the maximum number of bedrroms and bathrooms the user is looking for.

  Returns
  -------
  The users inputs.

  """
  while True:
    try:
      max_rooms = int(input('Enter the maximum number of bedrooms you\'re looking for: '))
      if max_rooms > 0:
        break
      else:
        print('Please enter a valid number of bedrooms')
        continue
    except:
      print('Please enter numbers')
      continue

  # Now get max bathrooms
  while True:
    try:
      max_baths = int(input('Enter the maximum number of bathrooms you\'re looking for: '))
      if max_baths > 0:
        break
      else:
        print('Please enter a valid number of bathrooms')
        continue
    except:
      print('Please enter numbers')
      continue

  return (max_rooms, max_baths)


In [ ]:
# Remove rows that dont fall within the rooms ranges
rooms = get_room_ranges()
max_bedrooms = rooms[0]
max_bathrooms = rooms[1]

for i, row in df.iterrows():
  num_bathrooms = int(row['# Bathrooms'])
  num_bedrooms = int(row['# Bedrooms'])

  if num_bedrooms > max_bedrooms:
    df.drop(i, inplace=True)
  elif num_bathrooms > max_bathrooms:
    df.drop(i, inplace=True)

df = df.reset_index(drop=True)
print_panda(df)

In [ ]:
# Install selenium in case it hasnt been installed yet
!pip install selenium
!pip install selenium webdriver-manager

# Using selenium to open dropdowns for web scraping:

Unfortunately, parts of some property's descriptions were hidden behind a 'Click to see more' dropdown button. To handle this, I had to learn had to use a webdriver with selenium. I set the webdriver's options so that it could more smoothly open and close, without error messages.

Then we go through each property, attempt to use the webdriver to click on the dropdown, and if it doesnt work then we can assume the full description is available without a dropdown click. Then we grab the full description paragraph at the appropriate tag and add it to the panda df.

In [ ]:
# Go through our panda and get the description for each property
# Create a new description column
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

# Configure Chrome options to run headless
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

for i, row in df.iterrows():
  print(f'Processing property at index {i}')
  url = row['Link']
  # Initialize the Chrome driver with the specified options
  driver = webdriver.Chrome(options=chrome_options)
  driver.get(url)
  # Stopping at the part of the description that no longer details the property
  stop_phrase = 'Click here to learn more about this listing.'
  try:
    # Go to the tag that contains the area we want the webdriver to click
    see_more = driver.find_element(By.XPATH, "//button[@class='btn btn-link d-block w-100']")
    see_more.click()

    html = driver.page_source
    page = BeautifulSoup(html, 'html.parser')
    # Now we can access the full description
    full_description = page.find(attrs={'class':'description-wrapper', 'data-v-1b462082':''})
    full_description = full_description.get_text().strip()
    description = full_description.split(stop_phrase)[0]
    df.loc[i, 'Description'] = description
  except:
    # We end up here if there was no dropdown. Then we know we can access the description without needing to click
    html = driver.page_source
    page = BeautifulSoup(html, 'html.parser')
    full_description = page.find(attrs={'class':'description-wrapper', 'data-v-1b462082':''})
    full_description = full_description.get_text().strip()
    description = full_description.split(stop_phrase)[0]
    df.loc[i, 'Description'] = description

  driver.close()
  print(f'Finished processing property at index {i}')

In [ ]:
print_panda(df)

In [ ]:
!pip install scikit-learn

In [ ]:
def users_keywords():
  """
  Gets the kinds of words and features the user is looking for in a property.

  Returns
  -------
  The user's inputs.

  """
  keywords = input('''Enter some keywords and features that youre looking for in a property.
  Example input: panoramic views stainless steel appliances pool patio nearby restaurants.
  ''')
  return keywords.lower()

# Go through the panda and compare the similarity between the user's string and the property's string
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

users_words = users_keywords()

for i, row in df.iterrows():
  prop_description = row['Description']
  # Use TfidVectorizer to transform the 2 strings into vectors
  vectorizer = TfidfVectorizer(stop_words='english') # remove common english words
  tfidf_matrix = vectorizer.fit_transform([users_words, prop_description])

  # Calculate the cosine similarity between the 2 vectors
  cosine_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])
  df.loc[i, 'Similarity'] = cosine_sim[0][0]


In [ ]:
if len(df) == 0:
  print('No matches found. Please restart the program')
else:
  df = df.sort_values(by=['Similarity'], ascending=False)
  df = df.reset_index(drop=True)
  top_three_matches = df.head(3)
  print_panda(top_three_matches)


# Time to print out everything from the top 3 matches!

In [ ]:
!pip install mistralai

In [ ]:
from google.colab import userdata
api_key=userdata.get('CSCI250')
from mistralai import Mistral
model = "mistral-large-latest"
client = Mistral(api_key=api_key)

def summarize_description(text: str):
    prompt = f"""Nicely summarize the property description that is in {text}."""
    chat_response = client.chat.complete(
       model=model,
       messages=[{"role":"user", "content":prompt}],
   )

    return chat_response.choices[0].message.content

In [ ]:
import cv2
from google.colab.patches import cv2_imshow
import urllib.request
import numpy as np
import textwrap

user_city = users_info[2]
user_city = user_city.replace('-', ' ')
user_city = user_city.title()
user_state = users_info[1]
num_matches = len(top_three_matches)
print(f'\033[1mYour Top {num_matches} in {user_city}, {user_state}:\033[0m \n')

address_list = []

# Go through and get the direct address for display
for i, row in top_three_matches.iterrows():
  link = row['Link']
  doc = requests.get(link)
  page = BeautifulSoup(doc.text, 'html.parser')
  address = page.find('div', class_='mb-2')
  address = address.text.strip()
  address = address + ' ' + user_city + ', ' + user_state
  address_list.append(address)

# Get all the info for each property
for i in range(len(top_three_matches)):
  print(f'\033[1m{i + 1})\033[0m \033[4m{address_list[i]}\033[0m:')
  print()
  price = top_three_matches.loc[i, 'Price']
  bedrooms = top_three_matches.loc[i, '# Bedrooms']
  bathrooms = top_three_matches.loc[i, '# Bathrooms']
  sq_ft = top_three_matches.loc[i, 'Sq. Ft.']
  print(f'''\033[1mDetails:\033[0m Price:  \033[4m{price}\033[0m   Bedrooms:\033 \033[4m{bedrooms}\033[0m'''
  f'''  Bathrooms: \033[4m{bathrooms}\033[0m   Sq. Ft.: \033[4m{sq_ft}\033[0m\n''')

  # Add in the description and the property's link
  description = top_three_matches.loc[i, 'Description']
  ai_summary = summarize_description(description)
  # Width of the textwrap
  #width = 160
  #wrapped_text = textwrap.fill(ai_summary, width)
  print('\033[1mProperty Description:\033[0m')
  print(ai_summary)
  print()

  # Web scrape the first 3 pictures of the property from the link
  url = top_three_matches.loc[i, 'Link']
  doc = requests.get(url)
  page = BeautifulSoup(doc.text, 'html.parser')
  for i in range(0, 3):
    pic_tag=page.find(attrs={'data-photoslide-img': str(i)})
    try:
      pic = pic_tag['src']
      req = urllib.request.urlopen(pic)
      arr = np.asarray(bytearray(req.read()), dtype=np.uint8)
      img = cv2.imdecode(arr, -1) # 'Load it as it is'
      # Save the image
      cv2.imwrite(f'Property{i}.png', img)
      # Display the image
      display_img = cv2.imread(f'Property{i}.png', img)
      cv2_imshow(display_img)
    except:
      print('Some property pics not found')

  print()
  print(f'\033[1mLink for more info:\033[0m {url}')
  print()
  print('---------------------------------------')